# VAD comparison across simulation chains

This notebook reuses the saved texts and BERT scores and computes NRC VAD v2.1 on the same texts, without generating rewrites or calling LLM APIs. It compares each step with the original news item and its actual input, and estimates the STDI resulting from replacing only VAD.

**Initial scope:** the first six chains: SSSS, CCCC, PPPP, DDDD, CCPP, and PPCC. The batch contains 16 chains; the filter below can be adjusted or removed.

**Scales:** BERT `(score - 1)/4`; NRC `(native_score + 1)/2`. Scores range from 0 to 1, and drift is the mean absolute difference across the three dimensions. The conversion is equivalent to dividing native NRC differences by 2 or differences on the 1-5 scale by 4. There is no normalization by the observed minimum/maximum. A common scale does not calibrate the models or prove equivalence.


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display
from plotly.subplots import make_subplots

from misinformation_simulation.analysis.vad_model_comparison import (
    DEFAULT_COMPARISON_CHAINS,
    run_vad_model_comparison,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RUNS_DIR = PROJECT_ROOT / "output/interaction_graph/app_runs"
LEXICON_PATH = PROJECT_ROOT / ".cache/nrc_vad/NRC-VAD-Lexicon-v2.1.zip"
OUTPUT_DIR = PROJECT_ROOT / "output/audit/SimulationVADModelComparison"
CHAIN_CODES = DEFAULT_COMPARISON_CHAINS  # Set to None to include all available chains.
COLORS = {"Saved BERT": "#2563eb", "NRC v2.1": "#e07a19"}
FIGURES = {}

if not LEXICON_PATH.exists():
    raise FileNotFoundError("Run uv run python scripts/audit_nrc_vad_context.py --download first.")

## 1. Reevaluate VAD only

Hypothetical STDI is computed only when BERT drift recalculated from saved scores and the historical formula match. Content and contradiction contributions are preserved. Missing values are not filled with neutrality. Historical BERT may have truncated texts; NRC processes full texts, and this difference has not been isolated.


In [ ]:
steps, summary, manifest = run_vad_model_comparison(
    RUNS_DIR,
    LEXICON_PATH,
    OUTPUT_DIR,
    chain_codes=CHAIN_CODES,
)
inventory = pd.read_csv(OUTPUT_DIR / "run_inventory.csv")
display(inventory)
print(
    f"Executions: {manifest['runs']}; news items: {manifest['news_items']}; "
    f"steps: {manifest['steps']}"
)
print(f"Unique texts scored locally: {manifest['unique_texts_scored']}")
print(f"BERT identity recorded in every step: {manifest['bert_model_id_recorded']}")
display(
    steps.groupby(["chain_code", "comparison_status", "formula_check_vs_original"])
    .size()
    .rename("steps")
    .reset_index()
)

## 2. VAD drift at each step in each chain

Lines show means for the same valid pairs at each step, with shared axes. The contrast **with the original** measures distance from the initial news item; the **incremental** contrast measures change relative to the text actually received at that step. Pair counts are available on hover. These curves do not measure belief or factual veracity.


In [ ]:
def show_figure(name, figure):
    figure.update_layout(template="plotly_white", font=dict(size=13), legend_title_text="Scorer")
    FIGURES[name] = figure
    display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))


for relation, label in [
    ("vs_original", "relative to the original"),
    ("incremental", "incremental"),
]:
    selected = summary.loc[(summary.metric == "vad_drift") & (summary.relation == relation)]
    long = selected.melt(
        id_vars=["run_id", "chain_code", "step_index", "paired_steps"],
        value_vars=["bert_mean", "nrc_mean"],
        var_name="model",
        value_name="mean_drift",
    )
    long["model"] = long.model.map({"bert_mean": "Saved BERT", "nrc_mean": "NRC v2.1"})
    figure = px.line(
        long,
        x="step_index",
        y="mean_drift",
        color="model",
        markers=True,
        line_group="run_id",
        facet_col="chain_code",
        facet_col_wrap=3,
        color_discrete_map=COLORS,
        hover_data=["run_id", "paired_steps"],
        title=f"VAD drift {label} by chain - paired means",
        labels={"step_index": "Step", "mean_drift": "VAD drift [0-1]", "model": "Scorer"},
    )
    figure.update_xaxes(dtick=1)
    figure.update_yaxes(range=[0, max(0.05, long.mean_drift.max() * 1.15)])
    figure.update_layout(height=280 * ((selected.chain_code.nunique() + 2) // 3))
    show_figure(f"vad_{relation}", figure)

## 3. Where do the scorers differ most?

Each cell shows **NRC - BERT** for mean drift or STDI at each step. Positive values indicate a larger result with NRC. The second figure preserves the saved non-affective STDI components. The color scale is symmetric around zero.


In [ ]:
for metric, label in [("vad_drift", "VAD drift"), ("stdi", "Hypothetical STDI")]:
    selected = summary.loc[(summary.metric == metric) & (summary.relation == "vs_original")]
    matrix = selected.pivot(index="run_id", columns="step_index", values="change_mean")
    run_labels = {
        row.run_id: f"{row.chain_code} (execution {index + 1})"
        for index, row in enumerate(
            selected[["run_id", "chain_code"]].drop_duplicates().itertuples()
        )
    }
    matrix.index = matrix.index.map(run_labels)
    bound = max(float(matrix.abs().max().max()), 1e-6)
    figure = px.imshow(
        matrix,
        text_auto=".4f",
        aspect="auto",
        color_continuous_scale="RdBu_r",
        range_color=[-bound, bound],
        title=f"{label}: NRC - BERT difference relative to the original",
        labels={"x": "Step", "y": "Execution", "color": "NRC - BERT"},
    )
    figure.update_layout(height=max(350, 55 * len(matrix)))
    show_figure(f"change_{metric}", figure)

## 4. Distributions of the same steps

The distribution below aggregates repeated steps from the same news items, so it should not be treated as a sample of independent observations. The scatter plot helps locate cases where the methods disagree; the diagonal indicates numerical equality, not accuracy.


In [ ]:
paired = steps.dropna(subset=["bert_vad_drift_vs_original", "nrc_vad_drift_vs_original"])
distribution = paired.melt(
    id_vars=["run_id", "chain_code", "news_id", "step_index"],
    value_vars=["bert_vad_drift_vs_original", "nrc_vad_drift_vs_original"],
    var_name="model",
    value_name="drift",
)
distribution["model"] = distribution.model.map(
    {
        "bert_vad_drift_vs_original": "Saved BERT",
        "nrc_vad_drift_vs_original": "NRC v2.1",
    }
)
figure = px.box(
    distribution,
    x="chain_code",
    y="drift",
    color="model",
    color_discrete_map=COLORS,
    title="VAD drift distribution by chain",
    labels={"chain_code": "Chain", "drift": "VAD drift [0-1]"},
)
show_figure("drift_distribution", figure)
figure = px.scatter(
    paired,
    x="bert_vad_drift_vs_original",
    y="nrc_vad_drift_vs_original",
    color="chain_code",
    opacity=0.45,
    hover_data=["metadata_title", "step_index", "nrc_rewritten_token_coverage"],
    labels={"bert_vad_drift_vs_original": "BERT drift", "nrc_vad_drift_vs_original": "NRC drift"},
    title="Paired VAD drift comparison - each point is one step",
)
limit = (
    float(paired[["bert_vad_drift_vs_original", "nrc_vad_drift_vs_original"]].max().max()) * 1.05
)
figure.add_shape(type="line", x0=0, y0=0, x1=limit, y1=limit, line=dict(color="gray", dash="dash"))
figure.update_xaxes(range=[0, limit])
figure.update_yaxes(range=[0, limit])
show_figure("paired_scatter", figure)

## 5. Valence, arousal, and dominance separately

Select a chain in the cell below and rerun this section. The charts show change magnitude; direction is shown in the individual example in the next section.


In [ ]:
SELECTED_CHAIN_CODE = "SSSS"
SELECTED_RUN_ID = None  # Use an exact run_id when multiple executions share one chain.
selected = summary.loc[summary.chain_code == SELECTED_CHAIN_CODE]
if SELECTED_RUN_ID is not None:
    selected = selected.loc[selected.run_id == SELECTED_RUN_ID]
if selected.empty:
    raise ValueError("The selected chain/run is outside the configured analysis scope.")
selected = selected.loc[
    (selected.relation == "vs_original")
    & (selected.metric != "stdi")
    & (selected.metric != "vad_drift")
]
long = selected.melt(
    id_vars=["run_id", "metric", "step_index", "paired_steps"],
    value_vars=["bert_mean", "nrc_mean"],
    var_name="model",
    value_name="drift",
)
long["model"] = long.model.map({"bert_mean": "Saved BERT", "nrc_mean": "NRC v2.1"})
figure = px.line(
    long,
    x="step_index",
    y="drift",
    color="model",
    markers=True,
    line_group="run_id",
    facet_col="metric",
    color_discrete_map=COLORS,
    title=f"Drift by dimension - {SELECTED_CHAIN_CODE}",
    labels={"step_index": "Step", "drift": "Normalized absolute difference"},
)
figure.update_xaxes(dtick=1)
show_figure("dimension_comparison", figure)
display(
    selected[
        ["run_id", "step_index", "metric", "paired_steps", "bert_mean", "nrc_mean", "change_mean"]
    ]
)

## 6. Follow a news item and read its texts

The cell below lists news items from the selected chain. Set `SELECTED_NEWS_ID` to inspect another case. **Step 0** represents the original news item. Higher scores in a dimension do not imply greater drift: drift depends on the distance from the reference text.


In [ ]:
example_pool = steps.loc[steps.chain_code == SELECTED_CHAIN_CODE]
if SELECTED_RUN_ID is None:
    SELECTED_RUN_ID = example_pool.run_id.iloc[0]
example_pool = example_pool.loc[example_pool.run_id == SELECTED_RUN_ID]
news_options = example_pool[["news_id", "metadata_title"]].drop_duplicates()
display(news_options)
SELECTED_NEWS_ID = news_options.news_id.iloc[0]  # Replace with a news_id from the table.
example = example_pool.loc[example_pool.news_id == SELECTED_NEWS_ID].sort_values("step_index")
figure = make_subplots(rows=1, cols=3, subplot_titles=["Valence", "Arousal", "Dominance"])
for column, dimension in enumerate(["valence", "arousal", "dominance"], start=1):
    for model, label in [("bert", "Saved BERT"), ("nrc", "NRC v2.1")]:
        values = [example.iloc[0][f"{model}_original_{dimension}_normalized"]]
        values += example[f"{model}_rewritten_{dimension}_normalized"].tolist()
        figure.add_trace(
            go.Scatter(
                x=[0] + example.step_index.tolist(),
                y=values,
                name=label,
                legendgroup=model,
                showlegend=column == 1,
                mode="lines+markers",
                connectgaps=False,
                line=dict(color=COLORS[label]),
            ),
            row=1,
            col=column,
        )
figure.update_yaxes(range=[0, 1], title_text="Normalized score")
figure.update_xaxes(title_text="Step (0 = original)", dtick=1)
figure.update_layout(title=f"News affective trajectory - {SELECTED_CHAIN_CODE}")
show_figure("individual_trajectory", figure)
columns = [
    "step_index",
    "node_label",
    "rewrite_status",
    "nrc_rewritten_token_coverage",
    "bert_vad_drift_vs_original",
    "nrc_vad_drift_vs_original",
    "bert_vad_drift_incremental",
    "nrc_vad_drift_incremental",
    "bert_stdi_vs_original",
    "nrc_stdi_vs_original",
    "stdi_change_vs_original",
    "bert_stdi_cumulative",
    "nrc_stdi_cumulative",
]
display(example[columns])
print("ORIGINAL:\n" + example.iloc[0].original_text)
for row in example.itertuples():
    print(f"\nSTEP {row.step_index} | {row.node_label} | {row.rewrite_status}")
    print(row.rewritten_text)

## 7. Coverage and largest disagreements

Low lexical coverage helps locate cases for inspection, but high coverage does not guarantee affective validity either. Examples are sorted by absolute STDI difference; missing values remain visible in the inventory.


In [ ]:
figure = px.histogram(
    steps,
    x="nrc_rewritten_token_coverage",
    color="chain_code",
    nbins=30,
    barmode="overlay",
    opacity=0.55,
    title="Lexical coverage of rewritten texts",
    labels={"nrc_rewritten_token_coverage": "Fraction of recognized tokens"},
)
figure.update_xaxes(range=[0, 1])
show_figure("lexical_coverage", figure)
inspection = steps.assign(abs_stdi_change=steps.stdi_change_vs_original.abs())
display(
    inspection.sort_values("abs_stdi_change", ascending=False)[
        [
            "chain_code",
            "news_id",
            "metadata_title",
            "step_index",
            "nrc_rewritten_token_coverage",
            "bert_stdi_vs_original",
            "nrc_stdi_vs_original",
            "stdi_change_vs_original",
        ]
    ].head(15)
)

## 8. Report and HTML version of the charts

The cumulative sum is not bounded by 1 and is not equivalent to distance from the original. The same topics may retain little affective contrast under both scorers. These analyses do not verify facts or assess belief, exposure, or sharing. More variation does not prove superiority; Portuguese has not been validated.

Simulation files remain intact. The output directory contains CSVs, a manifest, a report, and an HTML page with all charts, including the selected example. The HTML embeds Plotly and can be opened without an internet connection.


In [ ]:
display(Markdown((OUTPUT_DIR / "findings.md").read_text(encoding="utf-8")))
sections = []
for index, (_name, figure) in enumerate(FIGURES.items()):
    sections.append(figure.to_html(full_html=False, include_plotlyjs=True if index == 0 else False))
html = "<!doctype html><html lang='en'><meta charset='utf-8'><title>VAD Comparison</title>"
html += "<body style='max-width:1400px;margin:30px auto;font-family:sans-serif'>"
html += "<h1>VAD Comparison on Saved Simulation Texts</h1>"
html += (
    "<p>Normalized scores and drift. Hypothetical NRC STDI changes only VAD. "
    "Exploratory results; more variation does not demonstrate greater validity.</p>"
)
html += "<p>Chains: " + ", ".join(manifest["chain_codes"]) + "</p>"
html += "".join(sections) + "</body></html>"
(OUTPUT_DIR / "comparison_dashboard.html").write_text(html, encoding="utf-8")
print(f"Artifacts: {OUTPUT_DIR}")
print(f"Offline charts: {OUTPUT_DIR / 'comparison_dashboard.html'}")